In [1]:
from pathlib import Path
import pandas as pd

# notebook is inside notebooks/, so go one level up
ROOT = Path("..").resolve()
RAW = ROOT / "data" / "raw"

print("Project root:", ROOT)
print("Raw folder exists:", RAW.exists())
print("Files:", list(RAW.glob("*.csv")))

Project root: D:\Downloads\namaste_ankit_sql\ml_project_2
Raw folder exists: True
Files: [WindowsPath('D:/Downloads/namaste_ankit_sql/ml_project_2/data/raw/bureau_data.csv'), WindowsPath('D:/Downloads/namaste_ankit_sql/ml_project_2/data/raw/customers.csv'), WindowsPath('D:/Downloads/namaste_ankit_sql/ml_project_2/data/raw/loans.csv')]


In [2]:
customers = pd.read_csv(RAW / "customers.csv")
loans = pd.read_csv(RAW / "loans.csv")
bureau = pd.read_csv(RAW / "bureau_data.csv")

print("customers:", customers.shape)
print("loans     :", loans.shape)
print("bureau    :", bureau.shape)

customers: (50000, 12)
loans     : (50000, 15)
bureau    : (50000, 8)


Cell 3 — columns + first rows

In [ ]:
print("=== customers ===")
print(customers.columns.tolist())
display(customers.head())

=== customers ===
['cust_id', 'age', 'gender', 'marital_status', 'employment_status', 'income', 'number_of_dependants', 'residence_type', 'years_at_current_address', 'city', 'state', 'zipcode']


,cust_id,age,gender,marital_status,employment_status,income,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode
0,C00001,44,M,Married,Self-Employed,2586000,3,Owned,27,Delhi,Delhi,110001
1,C00002,38,M,Married,Salaried,1206000,3,Owned,4,Chennai,Tamil Nadu,600001
2,C00003,46,F,Married,Self-Employed,2878000,3,Owned,24,Kolkata,West Bengal,700001
3,C00004,55,F,Single,Self-Employed,3547000,1,Owned,15,Bangalore,Karnataka,560001
4,C00005,37,M,Married,Salaried,3432000,3,Owned,28,Pune,Maharashtra,411001


In [4]:
print("=== loans ===")
print(loans.columns.tolist())
display(loans.head())

=== loans ===
['loan_id', 'cust_id', 'loan_purpose', 'loan_type', 'sanction_amount', 'loan_amount', 'processing_fee', 'gst', 'net_disbursement', 'loan_tenure_months', 'principal_outstanding', 'bank_balance_at_application', 'disbursal_date', 'installment_start_dt', 'default']


,loan_id,cust_id,loan_purpose,loan_type,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,loan_tenure_months,principal_outstanding,bank_balance_at_application,disbursal_date,installment_start_dt,default
0,L00001,C00001,Auto,Secured,3004000,2467000,49340.0,444060,1973600,33,1630408,873386,2019-07-24,2019-08-10,False
1,L00002,C00002,Home,Secured,4161000,3883000,77660.0,698940,3106400,30,709309,464100,2019-07-24,2019-08-15,False
2,L00003,C00003,Personal,Unsecured,2401000,2170000,43400.0,390600,1736000,21,1562399,1476042,2019-07-24,2019-08-21,False
3,L00004,C00004,Personal,Unsecured,2345000,1747000,34940.0,314460,1397600,6,1257839,1031094,2019-07-24,2019-08-09,False
4,L00005,C00005,Auto,Secured,4647000,4520000,90400.0,813600,3616000,28,1772334,1032458,2019-07-24,2019-08-02,False


In [5]:
print("=== bureau ===")
print(bureau.columns.tolist())
display(bureau.head())

=== bureau ===
['cust_id', 'number_of_open_accounts', 'number_of_closed_accounts', 'total_loan_months', 'delinquent_months', 'total_dpd', 'enquiry_count', 'credit_utilization_ratio']


,cust_id,number_of_open_accounts,number_of_closed_accounts,total_loan_months,delinquent_months,total_dpd,enquiry_count,credit_utilization_ratio
0,C00001,1,1,42,0,0,3,7
1,C00002,3,1,96,12,60,5,4
2,C00003,2,1,82,24,147,6,58
3,C00004,3,0,115,15,87,5,26
4,C00005,4,2,120,0,0,5,10


Cell 4 — dtypes, missing, IDs

In [6]:
for name, df in [("customers", customers), ("loans", loans), ("bureau", bureau)]:
    print("\n========", name, "========")
    print(df.dtypes)
    print("\nMissing %:")
    print((df.isna().mean() * 100).round(2).sort_values(ascending=False).head(15))
    print("cust_id unique:", df["cust_id"].nunique() if "cust_id" in df.columns else "NO cust_id")


======== customers ========
cust_id                       str
age                         int64
gender                        str
marital_status                str
employment_status             str
income                      int64
number_of_dependants        int64
residence_type                str
years_at_current_address    int64
city                          str
state                         str
zipcode                     int64
dtype: object

Missing %:
residence_type              0.12
cust_id                     0.00
gender                      0.00
age                         0.00
marital_status              0.00
employment_status           0.00
income                      0.00
number_of_dependants        0.00
years_at_current_address    0.00
city                        0.00
state                       0.00
zipcode                     0.00
dtype: float64
cust_id unique: 50000

======== loans ========
loan_id                            str
cust_id                            str
l

Cleaning + first merge

In [7]:
print("gender:", customers["gender"].unique())
print("marital_status:", customers["marital_status"].unique())
print("employment_status:", customers["employment_status"].unique())
print("residence_type:", customers["residence_type"].unique())  # includes NaN
print("loan_purpose:", loans["loan_purpose"].unique())
print("loan_type:", loans["loan_type"].unique())
print("default counts:\n", loans["default"].value_counts())
print("default rate %:", round(loans["default"].mean() * 100, 2))

gender: <StringArray>
['M', 'F']
Length: 2, dtype: str
marital_status: <StringArray>
['Married', 'Single']
Length: 2, dtype: str
employment_status: <StringArray>
['Self-Employed', 'Salaried']
Length: 2, dtype: str
residence_type: <StringArray>
['Owned', 'Mortgage', 'Rented', nan]
Length: 4, dtype: str
loan_purpose: <StringArray>
['Auto', 'Home', 'Personal', 'Education', 'Personaal']
Length: 5, dtype: str
loan_type: <StringArray>
['Secured', 'Unsecured']
Length: 2, dtype: str
default counts:
 default
False    45703
True      4297
Name: count, dtype: int64
default rate %: 8.59


Cell B — numeric sanity (outliers)

In [8]:
print(customers[["age", "income", "number_of_dependants", "years_at_current_address"]].describe())

                age        income  number_of_dependants  \
count  50000.000000  5.000000e+04          50000.000000   
mean      39.550980  2.640898e+06              1.939540   
std        9.847752  2.629441e+06              1.535517   
min       18.000000  0.000000e+00              0.000000   
25%       33.000000  8.030000e+05              0.000000   
50%       40.000000  1.892000e+06              2.000000   
75%       46.000000  3.332250e+06              3.000000   
max       70.000000  1.199900e+07              5.000000   

       years_at_current_address  
count              50000.000000  
mean                  16.018440  
std                    8.926489  
min                    1.000000  
25%                    8.000000  
50%                   16.000000  
75%                   24.000000  
max                   31.000000  


In [9]:
print(loans[["loan_amount", "sanction_amount", "loan_tenure_months", "processing_fee"]].describe())


        loan_amount  sanction_amount  loan_tenure_months  processing_fee
count  5.000000e+04     5.000000e+04        50000.000000    5.000000e+04
mean   3.999679e+06     4.704828e+06           25.940520    8.049471e+04
std    5.376552e+06     6.267276e+06           12.433163    1.173123e+05
min    0.000000e+00     0.000000e+00            6.000000    0.000000e+00
25%    9.670000e+05     1.147000e+06           16.000000    1.934000e+04
50%    2.240000e+06     2.656000e+06           24.000000    4.480000e+04
75%    4.611000e+06     5.172250e+06           35.000000    9.224000e+04
max    4.781900e+07     5.217500e+07           59.000000    5.698030e+06


In [10]:
print(bureau.describe())

       number_of_open_accounts  number_of_closed_accounts  total_loan_months  \
count             50000.000000                50000.00000       50000.000000   
mean                  2.500140                    1.00106          76.127140   
std                   1.118725                    0.81412          43.762469   
min                   1.000000                    0.00000           1.000000   
25%                   1.000000                    0.00000          42.000000   
50%                   3.000000                    1.00000          71.000000   
75%                   4.000000                    2.00000         107.000000   
max                   4.000000                    2.00000         223.000000   

       delinquent_months     total_dpd  enquiry_count  \
count        50000.00000  50000.000000   50000.000000   
mean             4.87890     26.858000       5.009340   
std              5.85032     32.832832       2.029122   
min              0.00000      0.000000       1.0000

Cell C — the only missing column

In [11]:
print("residence_type missing rows:", customers["residence_type"].isna().sum())

residence_type missing rows: 62


In [12]:
display(customers[customers["residence_type"].isna()].head(10))

,cust_id,age,gender,marital_status,employment_status,income,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode
86,C00087,49,M,Single,Self-Employed,4770000,0,NaN,22,Bangalore,Karnataka,560001
1072,C01073,40,M,Married,Self-Employed,2068000,3,NaN,30,Bangalore,Karnataka,560001
1084,C01085,43,M,Married,Self-Employed,6833000,4,NaN,30,Lucknow,Uttar Pradesh,226001
3777,C03778,27,M,Single,Salaried,1398000,0,NaN,31,Jaipur,Rajasthan,302001
5915,C05916,31,M,Single,Salaried,2641000,0,NaN,16,Jaipur,Rajasthan,302001
6360,C06361,48,M,Married,Self-Employed,6811000,3,NaN,1,Bangalore,Karnataka,560001
6896,C06897,32,M,Single,Salaried,4991000,0,NaN,5,Bangalore,Karnataka,560001
7232,C07233,42,F,Married,Self-Employed,869000,4,NaN,19,Kolkata,West Bengal,700001
8030,C08031,35,F,Married,Salaried,197000,2,NaN,31,Hyderabad,Telangana,500001
8680,C08681,28,M,Single,Salaried,984000,0,NaN,14,Bangalore,Karnataka,560001


In [13]:
print(customers["residence_type"].value_counts(dropna=False))

residence_type
Owned       28238
Mortgage    11819
Rented       9881
NaN            62
Name: count, dtype: int64


In [15]:
## Use mode fill so we keep 50,000 IDs aligned across tables.

mode_res = customers["residence_type"].mode(dropna=True)[0]
print("Filling residence_type NA with:", mode_res)

Filling residence_type NA with: Owned


In [16]:
customers["residence_type"] = customers["residence_type"].fillna(mode_res)
print("NA left:", customers["residence_type"].isna().sum())

NA left: 0


Cell D — merge (inner join on cust_id)

In [ ]:
df = (
    loans
    .merge(customers, on="cust_id", how="inner")
    .merge(bureau, on="cust_id", how="inner")
)

In [18]:
print("merged shape:", df.shape)
print("unique cust_id:", df["cust_id"].nunique())
print("missing after merge:\n", df.isna().sum()[df.isna().sum() > 0])
df.head()

merged shape: (50000, 33)
unique cust_id: 50000
missing after merge:
 Series([], dtype: int64)


,loan_id,cust_id,loan_purpose,loan_type,sanction_amount,loan_amount,processing_fee,gst,net_disbursement,loan_tenure_months,...,city,state,zipcode,number_of_open_accounts,number_of_closed_accounts,total_loan_months,delinquent_months,total_dpd,enquiry_count,credit_utilization_ratio
0,L00001,C00001,Auto,Secured,3004000,2467000,49340.0,444060,1973600,33,...,Delhi,Delhi,110001,1,1,42,0,0,3,7
1,L00002,C00002,Home,Secured,4161000,3883000,77660.0,698940,3106400,30,...,Chennai,Tamil Nadu,600001,3,1,96,12,60,5,4
2,L00003,C00003,Personal,Unsecured,2401000,2170000,43400.0,390600,1736000,21,...,Kolkata,West Bengal,700001,2,1,82,24,147,6,58
3,L00004,C00004,Personal,Unsecured,2345000,1747000,34940.0,314460,1397600,6,...,Bangalore,Karnataka,560001,3,0,115,15,87,5,26
4,L00005,C00005,Auto,Secured,4647000,4520000,90400.0,813600,3616000,28,...,Pune,Maharashtra,411001,4,2,120,0,0,5,10


Cell E — save raw-clean snapshot (optional but useful)

In [19]:
from pathlib import Path

PROC = Path("..").resolve() / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

df.to_csv(PROC / "merged_raw.csv", index=False)
print("saved:", PROC / "merged_raw.csv")

saved: D:\Downloads\namaste_ankit_sql\ml_project_2\data\processed\merged_raw.csv


Fix typo + create the features the app uses

In [20]:
print(df["loan_purpose"].value_counts())

loan_purpose
Personal     17457
Home         15028
Auto          9936
Education     7557
Personaal       22
Name: count, dtype: int64


In [21]:
df["loan_purpose"] = df["loan_purpose"].replace({"Personaal": "Personal"})

In [22]:
print(df["loan_purpose"].value_counts())

loan_purpose
Personal     17479
Home         15028
Auto          9936
Education     7557
Name: count, dtype: int64


Cell 2 — target as 0/1 (easier for math later)

In [23]:
df["default"] = df["default"].astype(int)
print(df["default"].value_counts())

default
0    45703
1     4297
Name: count, dtype: int64


Cell 3 — features the Streamlit app will ask for

In [24]:
# 1) How big is the loan vs income?
df["loan_to_income"] = df["loan_amount"] / df["income"]

In [25]:
# 2) What % of the loan life was delinquent?
df["delinquency_ratio"] = df["delinquent_months"] / df["total_loan_months"]

In [26]:
# 3) Average days-past-due per delinquent month
#    if they never went delinquent, avg DPD = 0 (avoid divide-by-zero)
df["avg_dpd_per_delinquency"] = df["total_dpd"] / df["delinquent_months"].replace(0, pd.NA)
df["avg_dpd_per_delinquency"] = df["avg_dpd_per_delinquency"].fillna(0)

In [27]:
print(df[["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency",
          "credit_utilization_ratio", "number_of_open_accounts"]].describe())

print("NaN in new cols:",
      df[["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency"]].isna().sum().to_dict())

       loan_to_income  delinquency_ratio  credit_utilization_ratio  \
count    49992.000000       50000.000000               50000.00000   
mean         1.555239           0.103984                  43.36142   
std          0.971679           0.172995                  29.35598   
min          0.296170           0.000000                   0.00000   
25%          0.770145           0.000000                  18.00000   
50%          1.159964           0.037975                  39.00000   
75%          2.453737           0.130435                  67.00000   
max          4.591270           1.000000                  99.00000   

       number_of_open_accounts  
count             50000.000000  
mean                  2.500140  
std                   1.118725  
min                   1.000000  
25%                   1.000000  
50%                   3.000000  
75%                   4.000000  
max                   4.000000  
NaN in new cols: {'loan_to_income': 8, 'delinquency_ratio': 0, 'avg_dpd_

## Cell 4 — tiny EDA (only default rate by category)

In [28]:
def default_rate(col):
    return (
        df.groupby(col)["default"]
          .mean()
          .mul(100)
          .round(2)
          .sort_values(ascending=False)
    )

print("By residence_type\n", default_rate("residence_type"))
print("\nBy loan_purpose\n", default_rate("loan_purpose"))
print("\nBy loan_type\n", default_rate("loan_type"))
print("\nBy employment_status\n", default_rate("employment_status"))
print("\nBy gender\n", default_rate("gender"))

By residence_type
 residence_type
Rented      16.34
Mortgage     9.37
Owned        5.56
Name: default, dtype: float64

By loan_purpose
 loan_purpose
Home         15.52
Education     9.85
Personal      4.54
Auto          4.29
Name: default, dtype: float64

By loan_type
 loan_type
Secured      10.77
Unsecured     4.54
Name: default, dtype: float64

By employment_status
 employment_status
Salaried         9.17
Self-Employed    8.27
Name: default, dtype: float64

By gender
 gender
F    8.79
M    8.47
Name: default, dtype: float64


In [29]:
# Cell 5 — columns we will not feed the model (leakage / IDs)

In [30]:
id_cols = ["loan_id", "cust_id"]

# known only after money is already given
post_disbursal = [
    "principal_outstanding",
    "disbursal_date",
    "installment_start_dt",
    "net_disbursement",
    "gst",
    "processing_fee",
]

# raw pieces already replaced by ratios
raw_replaced = ["delinquent_months", "total_loan_months", "total_dpd"]

print("will drop later:", id_cols + post_disbursal + raw_replaced)

will drop later: ['loan_id', 'cust_id', 'principal_outstanding', 'disbursal_date', 'installment_start_dt', 'net_disbursement', 'gst', 'processing_fee', 'delinquent_months', 'total_loan_months', 'total_dpd']


In [31]:
#Cell 6 — save

In [32]:
from pathlib import Path
PROC = Path("..").resolve() / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
df.to_csv(PROC / "merged_with_features.csv", index=False)
print("saved", PROC / "merged_with_features.csv", "shape", df.shape)

saved D:\Downloads\namaste_ankit_sql\ml_project_2\data\processed\merged_with_features.csv shape (50000, 36)


In [33]:
print(df["loan_purpose"].value_counts())
print("residence NA:", df["residence_type"].isna().sum())

feats = ["loan_to_income", "delinquency_ratio", "avg_dpd_per_delinquency",
         "credit_utilization_ratio", "number_of_open_accounts"]
print(df[feats].describe())
print("NaN:", df[feats].isna().sum().to_dict())
print("inf loan_to_income:", (df["loan_to_income"] == float("inf")).sum())

def default_rate(col):
    return df.groupby(col)["default"].mean().mul(100).round(2).sort_values(ascending=False)

for c in ["residence_type", "loan_purpose", "loan_type", "employment_status", "gender"]:
    print(f"\n=== {c} ===")
    print(default_rate(c))

loan_purpose
Personal     17479
Home         15028
Auto          9936
Education     7557
Name: count, dtype: int64
residence NA: 0
       loan_to_income  delinquency_ratio  credit_utilization_ratio  \
count    49992.000000       50000.000000               50000.00000   
mean         1.555239           0.103984                  43.36142   
std          0.971679           0.172995                  29.35598   
min          0.296170           0.000000                   0.00000   
25%          0.770145           0.000000                  18.00000   
50%          1.159964           0.037975                  39.00000   
75%          2.453737           0.130435                  67.00000   
max          4.591270           1.000000                  99.00000   

       number_of_open_accounts  
count             50000.000000  
mean                  2.500140  
std                   1.118725  
min                   1.000000  
25%                   1.000000  
50%                   3.000000  
75%    

Cell — drop IDs + leakage, keep model columns


In [34]:
drop_cols = [
    "loan_id", "cust_id",
    "principal_outstanding", "disbursal_date", "installment_start_dt",
    "net_disbursement", "gst", "processing_fee",
    "delinquent_months", "total_loan_months", "total_dpd",
    "sanction_amount",   # almost the same info as loan_amount
]

model_df = df.drop(columns=drop_cols)
print(model_df.columns.tolist())
print(model_df.shape)
print(model_df.dtypes)

['loan_purpose', 'loan_type', 'loan_amount', 'loan_tenure_months', 'bank_balance_at_application', 'default', 'age', 'gender', 'marital_status', 'employment_status', 'income', 'number_of_dependants', 'residence_type', 'years_at_current_address', 'city', 'state', 'zipcode', 'number_of_open_accounts', 'number_of_closed_accounts', 'enquiry_count', 'credit_utilization_ratio', 'loan_to_income', 'delinquency_ratio', 'avg_dpd_per_delinquency']
(50000, 24)
loan_purpose                       str
loan_type                          str
loan_amount                      int64
loan_tenure_months               int64
bank_balance_at_application      int64
default                          int64
age                              int64
gender                             str
marital_status                     str
employment_status                  str
income                           int64
number_of_dependants             int64
residence_type                     str
years_at_current_address         int64
ci

In [35]:
model_df = model_df.drop(columns=["city", "state", "zipcode"])
print("final cols:", model_df.columns.tolist())
print(model_df.shape)

final cols: ['loan_purpose', 'loan_type', 'loan_amount', 'loan_tenure_months', 'bank_balance_at_application', 'default', 'age', 'gender', 'marital_status', 'employment_status', 'income', 'number_of_dependants', 'residence_type', 'years_at_current_address', 'number_of_open_accounts', 'number_of_closed_accounts', 'enquiry_count', 'credit_utilization_ratio', 'loan_to_income', 'delinquency_ratio', 'avg_dpd_per_delinquency']
(50000, 21)


In [36]:
from pathlib import Path
PROC = Path("..").resolve() / "data" / "processed"
model_df.to_csv(PROC / "model_table_v1.csv", index=False)